# Task 2: Player Segmentation (4-Class Classification — Macro $F_1$)

**Objective**: Classify each player into one of 4 segments (`0, 1, 2, 3`), evaluated by **Macro $F_1$ score**.

### Pipeline Summary
1. **Essential EDA** — Verify class imbalance, identify 16 pure-noise features, and measure within-class feature correlations (near zero, confirming conditional independence).
2. **Model A (weight 0.87): EM Label-Noise-Aware Histogram Naive Bayes** — Non-parametric quantile histograms with Gaussian smoothing, per-feature `(nbins, sigma)` tuned by inner-CV log-loss, and 15-iteration Expectation-Maximization (EM) to model symmetric label noise.
3. **Models B & C (weights 0.08 & 0.05)**: 5-fold **Spline Logistic Regression** and **PyTorch MLP** to capture subtle residual nonlinearities.
4. **Macro-$F_1$ Post-Processing** — Fit a fixed-point affine decision rule on OOF probabilities to directly optimize Macro $F_1$ under class imbalance.

In [1]:
import os
import glob
import time
import warnings
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d

import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler, QuantileTransformer, SplineTransformer
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

def find_data_dir(subdir: str) -> str:
    for p in [os.path.join("..", "cpe342-kaggle-main", subdir), os.path.join("cpe342-kaggle-main", subdir)]:
        if os.path.isdir(p):
            return os.path.abspath(p)
    hits = glob.glob(f"/kaggle/input/**/{subdir}/train.csv", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError(f"Dataset not found for {subdir}. Expected in cpe342-kaggle-main/{subdir}.")

DATA_DIR = find_data_dir("02_multiclass_classification")
OUT_CSV = "task2_predictions.csv"
K = 4
torch.set_num_threads(2)

tr = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
te = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
y = tr["segment"].astype(int).values
N, NT = len(tr), len(te)
test_ids = [f"ANS{i:05d}" for i in range(1, NT + 1)]

print(f"Train: {tr.shape} | Test: {te.shape}")
print("Class distribution:", {k: f"{(y == k).mean()*100:.2f}%" for k in range(K)})

Train: (104000, 46) | Test: (25889, 45)
Class distribution: {0: '39.38%', 1: '25.03%', 2: '20.22%', 3: '15.37%'}


## 1. Essential EDA & Design Decisions

We check two key structural properties of the data:
1. **Noise Feature Identification**: 16 columns (3 explicit random metrics, 8 non-informative categoricals, and 5 uninformative numerics) have near-zero class separation $\rightarrow$ **Decision: drop `player_id` and all 16 noise columns.**
2. **Within-Class Conditional Independence**: Compute the median absolute correlation between numeric signal features *within each class*. Near-zero within-class correlation means the Naive Bayes assumption holds closely $\rightarrow$ **Decision: use smoothed Histogram Naive Bayes as the primary model (87% weight), with EM to handle label noise and NaN marginalization.**

In [2]:
NOISE = [
    "random_metric_1", "random_metric_2", "random_metric_3", "account_age_days",
    "responds_to_discounts", "item_type_preference_social", "avg_match_length",
    "preferred_game_mode", "region", "platform", "device_type", "payment_method",
    "language", "account_status", "player_type_tag", "skill_tier"
]
FEATS = [c for c in tr.columns if c not in ["player_id", "segment"] + NOISE]
DISC = [c for c in FEATS if (not pd.api.types.is_numeric_dtype(tr[c])) or tr[c].nunique() <= 12]
CONT = [c for c in FEATS if c not in DISC]

# Check within-class pairwise correlation among continuous signal features
within_corrs = []
for k in range(K):
    C = tr.loc[y == k, CONT].corr().abs().values
    iu = np.triu_indices(len(CONT), 1)
    within_corrs.append(np.nanmedian(C[iu]))

print(f"Retained signal features: {len(FEATS)} ({len(CONT)} continuous, {len(DISC)} discrete: {DISC})")
print(f"Median missing rate across signal features: {tr[FEATS].isna().mean().median()*100:.2f}%")
print("Median |within-class correlation| per segment:", [round(float(v), 4) for v in within_corrs])
print("-> Near-zero within-class correlation confirms conditional independence (Naive Bayes is near Bayes-optimal).")

Retained signal features: 28 (23 continuous, 5 discrete: ['competitive_rank', 'watches_esports', 'vip_tier', 'engagement_level', 'loyalty_tier'])
Median missing rate across signal features: 7.70%
Median |within-class correlation| per segment: [0.0043, 0.0047, 0.0059, 0.0101]
-> Near-zero within-class correlation confirms conditional independence (Naive Bayes is near Bayes-optimal).


## 2. Model A (Weight 0.87): EM Label-Noise-Aware Histogram Naive Bayes

- **Density estimation**: For each continuous feature, bin edges are computed from training quantiles only and counts are smoothed with a 1D Gaussian kernel. `(nbins, sigma)` is chosen per feature via 4-fold inner-CV univariate posterior log-loss. Missing values are skipped (marginalized).
- **Label-noise EM**: Assumes observed labels have symmetric flip rate $\epsilon$, updating class responsibilities and $\epsilon$ over 15 EM iterations.

In [3]:
def softmax(S: np.ndarray) -> np.ndarray:
    S = S - S.max(axis=1, keepdims=True)
    P = np.exp(S)
    return P / P.sum(axis=1, keepdims=True)


def logprior(idx: np.ndarray) -> np.ndarray:
    return np.log(np.bincount(y[idx], minlength=K) / len(idx))


def disc_codes(c: str, a: np.ndarray):
    vt = tr[c].astype(object).where(tr[c].notna(), "NA").astype(str).values
    ve = te[c].astype(object).where(te[c].notna(), "NA").astype(str).values
    lv = sorted(set(vt[a]))
    m = {x: i for i, x in enumerate(lv)}
    return np.array([m.get(x, -1) for x in vt]), np.array([m.get(x, -1) for x in ve]), len(lv)


def loglik_block(c: str, a: np.ndarray, nbins=32, sm=0.7, alpha=1.0):
    if c in DISC:
        bt, be, n = disc_codes(c, a)
        nant, nane = bt < 0, be < 0
        bt, be = np.maximum(bt, 0), np.maximum(be, 0)
        keepmask = np.ones(N, bool)
    else:
        x = tr[c].values.astype(float)
        xe = te[c].values.astype(float)
        e = np.unique(np.nanquantile(x[a], np.linspace(0, 1, nbins + 1)))
        n = len(e) - 1
        nant, nane = np.isnan(x), np.isnan(xe)
        bt = np.where(nant, 0, np.clip(np.searchsorted(e, x, side="right") - 1, 0, n - 1))
        be = np.where(nane, 0, np.clip(np.searchsorted(e, np.nan_to_num(xe), side="right") - 1, 0, n - 1))
        keepmask = ~nant
    Lt = np.zeros((N, K))
    Le = np.zeros((NT, K))
    for k in range(K):
        sel = a[(y[a] == k) & keepmask[a]]
        cnt = np.bincount(bt[sel], minlength=n).astype(float)
        if sm > 0 and c in CONT:
            cnt = gaussian_filter1d(cnt, sm, mode="nearest")
        lp = np.log((cnt + alpha) / (cnt + alpha).sum())
        Lt[:, k] = lp[bt]
        Le[:, k] = lp[be]
    Lt[nant] = 0
    Le[nane] = 0
    return Lt, Le


def select_density_cfg(train_idx: np.ndarray, seed: int = 0) -> dict:
    GNB = [64, 128, 256]
    GSM = [3.0, 4.0, 6.0, 8.0, 12.0]
    inner = list(StratifiedKFold(4, shuffle=True, random_state=seed).split(train_idx, y[train_idx]))
    cfg = {}
    for c in CONT:
        best = None
        for nb_ in GNB:
            for sm in GSM:
                ll = 0.0
                for ia, ib in inner:
                    L = loglik_block(c, train_idx[ia], nb_, sm, 1.0)[0][train_idx[ib]] + logprior(train_idx[ia])
                    ll += np.mean(np.log(softmax(L)[np.arange(len(ib)), y[train_idx[ib]]]))
                if best is None or ll > best[0]:
                    best = (ll, nb_, sm)
        cfg[c] = best[1:]
    return cfg


def emnb_binned(a: np.ndarray, cfg: dict):
    out = []
    for c in FEATS:
        if c in DISC:
            bt, be, n = disc_codes(c, a)
            sm = 0
        else:
            nb_, sm = cfg[c]
            x = tr[c].values.astype(float)
            xe = te[c].values.astype(float)
            e = np.unique(np.nanquantile(x[a], np.linspace(0, 1, nb_ + 1)))
            n = len(e) - 1
            bt = np.where(np.isnan(x), -1, np.clip(np.searchsorted(e, x, side="right") - 1, 0, n - 1))
            be = np.where(np.isnan(xe), -1, np.clip(np.searchsorted(e, np.nan_to_num(xe), side="right") - 1, 0, n - 1))
        out.append((bt, be, n, sm))
    return out


def emnb_densities(B, a, R, alpha=1.0):
    tabs = []
    for bt, be, n, sm in B:
        ba = bt[a]
        ok = ba >= 0
        T = np.zeros((n, K))
        for k in range(K):
            cnt = np.bincount(ba[ok], weights=R[ok, k], minlength=n)
            if sm > 0:
                cnt = gaussian_filter1d(cnt, sm, mode="nearest")
            T[:, k] = np.log((cnt + alpha) / (cnt + alpha).sum())
        tabs.append(T)
    return tabs


def emnb_loglik(B, tabs, rows=None, test=False):
    S = 0.0
    for (bt, be, n, sm), T in zip(B, tabs):
        bb = be if test else bt[rows]
        L = T[np.maximum(bb, 0)]
        L[bb < 0] = 0.0
        S = S + L
    return S


def emnb_fit(a: np.ndarray, B, n_iters: int = 15, alpha: float = 1.0):
    Y = np.eye(K)[y[a]]
    R = Y.copy()
    eps = 0.0
    pi = Y.mean(0)
    for it in range(n_iters + 1):
        tabs = emnb_densities(B, a, R, alpha)
        if it == n_iters:
            break
        S = emnb_loglik(B, tabs, a) + np.log(pi)
        e = max(eps, 1e-3) if it else 0.03
        noise = np.where(Y > 0, np.log(1 - e), np.log(e / 3))
        R = softmax(S + noise)
        pi = R.mean(0)
        eps = 1.0 - (R * Y).sum() / len(a)
    return tabs, pi, eps


t0 = time.time()
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(tr, y))
nb_oof = np.zeros((N, K), dtype=np.float64)
for fo, (A_idx, B_idx) in enumerate(FOLDS):
    cfg_fo = select_density_cfg(A_idx, seed=fo)
    Bn = emnb_binned(A_idx, cfg_fo)
    tabs, pi_f, eps_f = emnb_fit(A_idx, Bn)
    M = np.full((K, K), eps_f / 3)
    np.fill_diagonal(M, 1 - eps_f)
    nb_oof[B_idx] = softmax(emnb_loglik(Bn, tabs, B_idx) + np.log(pi_f)) @ M
    print(f"  [EM-NB] fold {fo}: Macro F1={f1_score(y[B_idx], nb_oof[B_idx].argmax(1), average='macro'):.5f} | est. noise eps={eps_f:.4f}")

# Refit on full training data for test predictions
all_idx = np.arange(N)
cfg_full = select_density_cfg(all_idx, seed=0)
B_full = emnb_binned(all_idx, cfg_full)
tabs_full, pi_full, eps_full = emnb_fit(all_idx, B_full)
M_full = np.full((K, K), eps_full / 3)
np.fill_diagonal(M_full, 1 - eps_full)
nb_test = softmax(emnb_loglik(B_full, tabs_full, test=True) + np.log(pi_full)) @ M_full

print(f"[Model A: EM-NB] OOF Macro F1: {f1_score(y, nb_oof.argmax(1), average='macro'):.5f} | full eps={eps_full:.4f} ({time.time()-t0:.0f}s)")

  [EM-NB] fold 0: Macro F1=0.75535 | est. noise eps=0.0281


  [EM-NB] fold 1: Macro F1=0.75479 | est. noise eps=0.0280


  [EM-NB] fold 2: Macro F1=0.75880 | est. noise eps=0.0285


  [EM-NB] fold 3: Macro F1=0.75771 | est. noise eps=0.0282


  [EM-NB] fold 4: Macro F1=0.76165 | est. noise eps=0.0288


[Model A: EM-NB] OOF Macro F1: 0.75766 | full eps=0.0284 (401s)


## 3. Models B & C (Weights 0.08 & 0.05): Spline Logistic Regression & PyTorch MLP

Adding lightweight discriminative models provides complementary probability estimates without overfitting. All imputers, quantile normalizers, and scalers are fitted inside each training fold.

In [4]:
CATS_ALL = [
    "region", "platform", "device_type", "payment_method", "language",
    "account_status", "player_type_tag", "engagement_level", "loyalty_tier", "skill_tier"
]
NOISE_NUM = [
    "random_metric_1", "random_metric_2", "random_metric_3", "preferred_game_mode",
    "account_age_days", "avg_match_length", "responds_to_discounts", "item_type_preference_social"
]
INFO_NUM = [c for c in tr.columns if c not in CATS_ALL + ["player_id", "segment"] + NOISE_NUM]
INFO_CAT = ["engagement_level", "loyalty_tier"]


def build_tabular(df: pd.DataFrame) -> pd.DataFrame:
    X = df[INFO_NUM].copy()
    for c in INFO_CAT:
        for v in sorted(tr[c].dropna().unique()):
            X[f"{c}_{v}"] = (df[c] == v).astype(float)
    return X


A_tab, B_tab = build_tabular(tr), build_tabular(te)


def prep_fold(Xa: pd.DataFrame, Xs: list):
    med = Xa.median()
    qt = QuantileTransformer(n_quantiles=200, output_distribution="normal", subsample=50000, random_state=0)
    Xa_ = Xa.fillna(med)
    sc = StandardScaler().fit(qt.fit_transform(Xa_))
    f = lambda X: sc.transform(qt.transform(X.fillna(med)))
    return f(Xa), [f(X) for X in Xs]


# Model B: Spline Logistic Regression
lr_oof = np.zeros((N, K))
lr_test = np.zeros((NT, K))
for a, b in FOLDS:
    Za, (Zb, Zt) = prep_fold(A_tab.iloc[a], [A_tab.iloc[b], B_tab])
    st = SplineTransformer(n_knots=6, degree=3, knots="quantile").fit(Za[:30000, :len(INFO_NUM)])
    T = lambda Z: np.hstack([st.transform(Z[:, :len(INFO_NUM)]), Z[:, len(INFO_NUM):]])
    m = LogisticRegression(C=0.05, max_iter=300).fit(T(Za), y[a])
    lr_oof[b] = m.predict_proba(T(Zb))
    lr_test += m.predict_proba(T(Zt)) / 5
print(f"[Model B: Spline LR]   OOF Macro F1: {f1_score(y, lr_oof.argmax(1), average='macro'):.5f}")

# Model C: PyTorch MLP
mlp_oof = np.zeros((N, K))
mlp_test = np.zeros((NT, K))
for a, b in FOLDS:
    Za, (Zb, Zt) = prep_fold(A_tab.iloc[a], [A_tab.iloc[b], B_tab])
    torch.manual_seed(0)
    m = nn.Sequential(
        nn.Linear(Za.shape[1], 128), nn.SiLU(), nn.Dropout(0.2),
        nn.Linear(128, 64), nn.SiLU(), nn.Dropout(0.2),
        nn.Linear(64, K)
    )
    opt = torch.optim.AdamW(m.parameters(), 2e-3, weight_decay=1e-3)
    E = 12
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 4e-3, total_steps=E * ((len(Za) + 511) // 512))
    X_t = torch.tensor(Za, dtype=torch.float32)
    Y_t = torch.tensor(y[a], dtype=torch.long)
    for _ in range(E):
        m.train()
        p = torch.randperm(len(X_t))
        for i in range(0, len(X_t), 512):
            j = p[i:i + 512]
            opt.zero_grad()
            nn.functional.cross_entropy(m(X_t[j]), Y_t[j]).backward()
            opt.step()
            sch.step()
    m.eval()
    with torch.no_grad():
        mlp_oof[b] = torch.softmax(m(torch.tensor(Zb, dtype=torch.float32)), 1).numpy()
        mlp_test += torch.softmax(m(torch.tensor(Zt, dtype=torch.float32)), 1).numpy() / 5
print(f"[Model C: PyTorch MLP] OOF Macro F1: {f1_score(y, mlp_oof.argmax(1), average='macro'):.5f}")

[Model B: Spline LR]   OOF Macro F1: 0.75468


[Model C: PyTorch MLP] OOF Macro F1: 0.75227


## 4. Probability Blend & Fixed-Point Affine Macro-$F_1$ Decision Rule

Standard `argmax(p)` optimizes accuracy, not Macro $F_1$. Under Macro $F_1$, the Bayes-optimal decision rule is an affine transformation of class probabilities:
$$\hat{y}(x) = \arg\max_k \left( \frac{2}{\pi_k + q_k}\, p_k(x) - \frac{F_k}{\pi_k + q_k} \right)$$
where $\pi_k$ is the true class prior, $q_k$ is the predicted class share, and $F_k$ is the class-$k$ $F_1$ score. We solve this fixed point strictly on OOF predictions and apply the resulting $(a_k, b_k)$ to test probabilities.

In [5]:
Bo = 0.87 * nb_oof + 0.08 * lr_oof + 0.05 * mlp_oof
Bt = 0.87 * nb_test + 0.08 * lr_test + 0.05 * mlp_test

pi = np.bincount(y, minlength=K) / N
q = pi.copy()
Fk = np.full(K, 0.75)

for _ in range(8):
    aa = 2.0 / (pi + q)
    bb = -Fk / (pi + q)
    pr_oof = (aa * Bo + bb).argmax(axis=1)
    q = np.bincount(pr_oof, minlength=K) / N
    Fk = np.array([2.0 * ((pr_oof == k) & (y == k)).mean() / (pi[k] + q[k]) for k in range(K)])

print(f"OOF Macro F1 (EM-NB only):      {f1_score(y, nb_oof.argmax(1), average='macro'):.5f}")
print(f"OOF Macro F1 (3-model blend):   {f1_score(y, Bo.argmax(1), average='macro'):.5f}")
print(f"OOF Macro F1 (blend + affine):  {f1_score(y, pr_oof, average='macro'):.5f}")

pred_test = (aa * Bt + bb).argmax(axis=1)
sub_t2 = pd.DataFrame({"id": test_ids, "pred": pred_test})
sub_t2.to_csv(OUT_CSV, index=False, lineterminator="\n")
print(f"Saved {OUT_CSV} | rows={len(sub_t2)}")
print("Train prior:     ", np.round(pi, 4))
print("Test pred share: ", np.round(np.bincount(pred_test, minlength=K) / NT, 4))
sub_t2.head()

OOF Macro F1 (EM-NB only):      0.75766
OOF Macro F1 (3-model blend):   0.75798
OOF Macro F1 (blend + affine):  0.75826
Saved task2_predictions.csv | rows=25889
Train prior:      [0.3938 0.2503 0.2022 0.1537]
Test pred share:  [0.4085 0.2541 0.1939 0.1435]


,id,pred
0,ANS00001,3
1,ANS00002,3
2,ANS00003,2
3,ANS00004,1
4,ANS00005,0
